# Basic fitting in **EasyDynamics**
We here show how to use **EasyDynamics** to fit a simple synthetic data set. We start with the simplest possible example; later tutorials will be more involved.

The general procedure is to create an `Experiment` object to hold the data, a `SampleModel` to describe the model, and `Analysis` to fit the model to the data.

In [1]:
# Imports
import pooch

from easydynamics.analysis.analysis import Analysis
from easydynamics.experiment import Experiment
from easydynamics.sample_model import BrownianTranslationalDiffusion
from easydynamics.sample_model import ComponentCollection
from easydynamics.sample_model import DeltaFunction, DampedHarmonicOscillator
from easydynamics.sample_model import Gaussian
from easydynamics.sample_model import Lorentzian
from easydynamics.sample_model import Polynomial
from easydynamics.sample_model.background_model import BackgroundModel
from easydynamics.sample_model.instrument_model import InstrumentModel
from easydynamics.sample_model.resolution_model import ResolutionModel
from easydynamics.sample_model.sample_model import SampleModel
import numpy as np
import matplotlib.pyplot as plt
import scipp as sc
# Make the plots interactive
%matplotlib widget

We first create an `Experiment` object to contain the data. The data must either be a `hdf5` file or a `scipp.DataArray`; in both cases it must have coordinates `Q` and `energy`. We here use Pooch to download an example data set.

<details>
  <summary><strong>💡 Tip</strong></summary>
  <div style="padding:10px; margin-top:5px; border-left:4px solid #4caf50; background:#e8f5e9;">
    If you have one or more data files that you want to fit using EasyDynamics, you can reach out to us e.g. at henrik.jacobsen@ess.eu for help.
  </div>
</details>

We give the `Experiment` a `display_name` which is used as the title when plotting the data.

In [2]:
import scipp as sc
import plopp as pp
import numpy as np
Q=sc.linspace(start=0.2,stop=2.1,num=16,unit='1/angstrom', dim='Q')

component_collection=ComponentCollection()
component_collection.append_component(Gaussian(area=0.45,width=0.1))
# component_collection.append_component(Lorentzian(area=1.45,width=0.2,center=-1.5))
# component_collection.append_component(DampedHarmonicOscillator(area=1.45,width=0.1,center=0.4))

model=SampleModel(components=component_collection,Q=Q)


for i in range(Q.size):
    components=model.get_component_collection(i)
    # offset = sc.scalar(value=np.random.uniform(-0.05,0.05), unit='meV')
    # offset = np.random.uniform(-0.025,0.025)
    offset = 0.0
    components.components[0].area=3.79-0.2*Q[i].value
    # components.components[1].center=-1.5+Q[i].value**2/10.0
    # components.components[2].center=0.4+Q[i].value/10.0
    # components.components[2].area=1.45+Q[i].value/10.0
    # components.components[0].center=offset
    # components.components[1].center.value+=offset
    # components.components[2].center.value+=offset

energy=sc.linspace(start=-3.0,stop=3.0,num=256,unit='meV',dim='energy')

intensity=sc.array(values=model.evaluate(x=energy),dims=['Q','energy'])

intensity_dataarray=sc.DataArray(data=intensity,coords={'Q':Q,'energy':energy})

noise = np.random.normal(loc=0.0, scale=0.15, size=intensity_dataarray.shape)
intensity_dataarray.values += noise

sc.io.save_hdf5(intensity_dataarray,'data/fake_simple_data.hdf5')
pp.slicer(intensity_dataarray)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [3]:
# Load the data
experiment = Experiment(display_name='Tutorial')

# file_path = pooch.retrieve(
#     url='https://github.com/easyscience/dynamics-lib/raw/refs/heads/master/docs/docs/tutorials/data/vanadium_data_example.h5',
#     known_hash='16cc1b327c303feeb88fb9dda5390dc4880b62396b1793f98c6fef0b27c7b873',
# )

file_path='data/fake_simple_data.hdf5'

experiment.load_hdf5(filename=file_path)

We can visualize the data in multiple ways, relying on plopp: https://scipp.github.io/plopp/

For now, let us plot the data using a slicer showing intensity as function of `energy` for various `Q`. You can dragg the slider to choose which $Q$ is displayed. Notice that the title is the `display_name` that we gave the `Experiment`.

<details>
  <summary><strong>💡 Tip</strong></summary>
  <div style="padding:10px; margin-top:5px; border-left:4px solid #4caf50; background:#e8f5e9;">
    You can change the title by running `experiment.display_name="new title"`
  </div>
</details>

In [4]:
experiment.plot_data(slicer=True)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

We now want to fit this data. The first step is to figure out what model to use. 

We observe a strong central peak that looks like a `Gaussian`, a pair of symmetric peaks that could be described by a `DampedHarmonicOscillator` (DHO) and a peak at negative energies with longer tails that is probably a `Lorentzian`. There seems to be no background far away from the peaks. The peaks seem to move around and change intensity with `Q`. The goal is to fit the data and extract the parameters describing these changes. 

The first step is to set up a model to describe these peaks. We want to define the three functions just mentioned, and put them in a single object called a `ComponentCollection`.

The `Gaussian` seems to be centered at 0, while the DHO peaks around $\pm 0.5$ meV and the Lorentzian center is around $-1.5$ meV. 

We define the three model components as follows. We could also give them a `unit` argument, but when it is left out, the default is `meV`.


In [5]:
gaussian=Gaussian(display_name='Gaussian', area=1, width=0.05)
DHO = DampedHarmonicOscillator(display_name='DHO', area=0.3, width=0.1,center=0.5)
lorentzian = Lorentzian(display_name='Lorentzian', area=0.5, width=0.2,center=-1.5)


We now create the `ComponentCollection`, which essentially is a list of the three components. 

In [6]:

component_collection = ComponentCollection()
component_collection.append_component(gaussian)
component_collection.append_component(DHO)
component_collection.append_component(lorentzian)

Let us have a look at out `ComponentCollection`. We can get an overview of the components in it like this:

In [7]:
print(component_collection)

<ComponentCollection unique_name='ComponentCollection_18' | Components: Gaussian_17, DampedHarmonicOscillator_0, Lorentzian_0>


To get more information about what's in the `ComponentCollection`, we can also list its components:

In [8]:
component_collection.components

[Gaussian(unique_name = Gaussian_17, unit = meV,
              area = <Parameter 'Gaussian area': 1.0000 meV, bounds=[0.0:inf]>,
  center = <Parameter 'Gaussian center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
  width = <Parameter 'Gaussian width': 0.0500 meV, bounds=[1e-10:inf]>),
 DampedHarmonicOscillator(display_name = DHO, unit = meV,
          area = <Parameter 'DHO area': 0.3000 meV, bounds=[0.0:inf]>,
  center = <Parameter 'DHO center': 0.5000 meV, bounds=[1e-10:inf]>,
  width = <Parameter 'DHO width': 0.1000 meV, bounds=[1e-10:inf]>),
 Lorentzian(unique_name = Lorentzian_0, unit = meV,
              area = <Parameter 'Lorentzian area': 0.5000 meV, bounds=[0.0:inf]>,
  center = <Parameter 'Lorentzian center': -1.5000 meV, bounds=[-inf:inf]>,
  width = <Parameter 'Lorentzian width': 0.2000 meV, bounds=[1e-10:inf]>)]

To access a particular component from the collection, we can use its index. Soon, it will also be possible to refer to components by name. So, for example, if we want to access the `Gaussian`, it is the first component in the collection, so we use the index 0:

In [9]:
component_collection.components[0]

Gaussian(unique_name = Gaussian_17, unit = meV,
             area = <Parameter 'Gaussian area': 1.0000 meV, bounds=[0.0:inf]>,
 center = <Parameter 'Gaussian center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 width = <Parameter 'Gaussian width': 0.0500 meV, bounds=[1e-10:inf]>)

We can access and change specific `Parameter`s of each component as we please. For example, to change the area of the `Gaussian` to 1.1 meV, we do this

In [10]:
component_collection.components[0].area = 1.1
component_collection.components[0]

Gaussian(unique_name = Gaussian_17, unit = meV,
             area = <Parameter 'Gaussian area': 1.1000 meV, bounds=[0.0:inf]>,
 center = <Parameter 'Gaussian center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 width = <Parameter 'Gaussian width': 0.0500 meV, bounds=[1e-10:inf]>)

Finally, we may want to visualize our `ComponentCollection`. We have not yet built in methods to plot `ComponentCollection`s or `ModelComponent`s directly, so we use scipp https://scipp.github.io/index.html. 

Note: you can also use numpy and matplotlib, if you are more familiar with them.


In [11]:
energy=sc.linspace(start=-3.0,stop=3.0,num=1000,unit='meV',dim='energy')

intensity=sc.array(values=component_collection.evaluate(x=energy),dims=['energy'],)

intensity_array = sc.DataArray(data=intensity,coords={'energy':energy})

sc.plot(intensity_array)


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

We would like to fit our data to this model. However, with around 20 Q values we do not want to copy/paste our fit function over and over again. Instead, EasyDynamics handles this for us; we simply have to create a `SampleModel` and pass it our collection of components.

In [12]:
model = SampleModel(components=component_collection)

Our model does not yet have any $Q$ values. We could tell it what $Q$ is simply by using the `Q` attribute.
```python
Q_values=sc.linspace(start=0.2,stop=2.1,num=16,unit='1/angstrom', dim='Q')

model.Q=Q_values
```

This would generate a copy of the `ComponentCollection` for each of the values of `Q`. 

However, since we want the `Q` values to match those of the experiment, it is much easier to let EasyDynamics handle it. To do this, we make an `Analysis` object and give it out experiment and sample model:

In [13]:
analysis = Analysis(
    experiment=experiment,
    sample_model=model,
)

A lot happens under the hood in this step. It automatically extracts the `Q` values from the experiment and passes them to our `SampleModel`, which in turn generates a copy of the `ComponentCollection` for each `Q`. We furthermore create an `Analysis1d` object for each `Q`. You will usually not need to think too hard about any of this, since you will mostly interact with your data and model through the `Analysis` object, but it can be good to know what is going on.

We can access the `Analysis1d` objects like this:


In [14]:
analysis.analysis_list

[ Analysis1d  (display_name=MyAnalysis_Q0,         unique_name=Analysis_0_Q0),
  Analysis1d  (display_name=MyAnalysis_Q1,         unique_name=Analysis_0_Q1),
  Analysis1d  (display_name=MyAnalysis_Q2,         unique_name=Analysis_0_Q2),
  Analysis1d  (display_name=MyAnalysis_Q3,         unique_name=Analysis_0_Q3),
  Analysis1d  (display_name=MyAnalysis_Q4,         unique_name=Analysis_0_Q4),
  Analysis1d  (display_name=MyAnalysis_Q5,         unique_name=Analysis_0_Q5),
  Analysis1d  (display_name=MyAnalysis_Q6,         unique_name=Analysis_0_Q6),
  Analysis1d  (display_name=MyAnalysis_Q7,         unique_name=Analysis_0_Q7),
  Analysis1d  (display_name=MyAnalysis_Q8,         unique_name=Analysis_0_Q8),
  Analysis1d  (display_name=MyAnalysis_Q9,         unique_name=Analysis_0_Q9),
  Analysis1d  (display_name=MyAnalysis_Q10,         unique_name=Analysis_0_Q10),
  Analysis1d  (display_name=MyAnalysis_Q11,         unique_name=Analysis_0_Q11),
  Analysis1d  (display_name=MyAnalysis_Q12,     


For now, let us use the `plot_data_and_model()` method to, well, plot our data and model! We can again use the slider to look at different $Q$.

In [15]:
analysis.plot_data_and_model()

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

The model is of course not yet particularly good. However, we can now start fitting it.

There are two ways to fit data in EasyDynamics: `independent` and `simultaneous`. The `independent` method treats each `Q` position as independent from the others and fits them one after the other, or fits a particular index as we shall show below. The `simultaneous` method is used when some parameters are shared between different Q. We do not use `simultaneous` fitting in this tutorial, but refer to the other tutorials.

Let us first fit a single Q index and plot the data and model to see how it looks. For this, we use the `independent` fit method and choose an arbitrary Q index

In [16]:
fit_result_independent_single_Q = analysis.fit(fit_method='independent', Q_index=5)
analysis.plot_data_and_model(Q_index=5)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

The fit looks very good. We can get the parameters for this fit by accesing the corresponding `Analysis1d` object:

In [17]:
analysis.analysis_list[5].get_all_parameters()

[<Parameter 'Gaussian area': 3.3335 ± 0.3795 meV, bounds=[0.0:inf]>,
 <Parameter 'Gaussian center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'Gaussian width': 0.0997 ± 0.0016 meV, bounds=[1e-10:inf]>,
 <Parameter 'DHO area': 0.3201 ± 0.3668 meV, bounds=[0.0:inf]>,
 <Parameter 'DHO center': 0.2130 ± 0.2646 meV, bounds=[1e-10:inf]>,
 <Parameter 'DHO width': 0.2799 ± 0.6906 meV, bounds=[1e-10:inf]>,
 <Parameter 'Lorentzian area': 1.697e-05 ± 2.137e+01 meV, bounds=[0.0:inf]>,
 <Parameter 'Lorentzian center': -1.5410 ± 960.9473 meV, bounds=[-inf:inf]>,
 <Parameter 'Lorentzian width': 0.0001 ± 140.8806 meV, bounds=[1e-10:inf]>,
 <Parameter 'energy_offset': -0.0002 ± 0.0005 meV, bounds=[-inf:inf]>]

Since the fit looked good, we can now fit all $Q$. We also plot the result, again using the slicer.

In [18]:
fit_result_independent_all_Q = analysis.fit(fit_method='independent')
analysis.plot_data_and_model()

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

It can be nice to inspect the fit parameters, and sometimes continue working with them. However, even in this relatively simple example, there are a lot of parameters. Accessing them individually as before will be a pain. Therefore, we can convert them to a scipp dataset.



In [19]:
analysis.parameters_to_dataset()

<scipp.Dataset>
Dimensions: Sizes[Q:16, ]
Coordinates:
* Q                         float64           [1/Å]  (Q)  [0.2, 0.326667, ..., 1.97333, 2.1]
Data:
  DHO area                  float64            [meV]  (Q)  [0.0144348, 0.158006, ..., 3.32168e-11, 0.0580512]  [0.00233663, 0.00318351, ..., 0.00108855, 19.9279]
  DHO center                float64            [meV]  (Q)  [0.0570637, 1.54502, ..., 0.347411, 1.31398]  [0.0031465, 0.103062, ..., 9.1136e+15, 8.6803e+08]
  DHO width                 float64            [meV]  (Q)  [0.0127598, 0.559552, ..., 0.475757, 16.1103]  [0.00422865, 0.15276, ..., 6.69309e+16, 5.19694e+11]
  Gaussian area             float64            [meV]  (Q)  [3.72565, 3.72931, ..., 3.41987, 3.32648]  [0.00255962, 0.000319491, ..., 0.0324494, 0.0169749]
  Gaussian center           float64            [meV]  (Q)  [0, 0, ..., 0, 0]  [0, 0, ..., 0, 0]
  Gaussian width            float64            [meV]  (Q)  [0.100007, 0.100552, ..., 0.100337, 0.100697]  [7.39517e-07, 2.60756e-07, ..., 5.05803e-06, 1.71234e-06]
  Lorentzian area           float64            [meV]  (Q)  [0.125095, 0.00509856, ..., 0.000652819, 0.482297]  [125834, 0.000186873, ..., 2.89991e+06, 14.813]
  Lorentzian center         float64            [meV]  (Q)  [-1.64928, -1.42574, ..., -1.58775, 0.275395]  [0.000411258, 0.00372033, ..., 2024.61, 115.19]
  Lorentzian width          float64            [meV]  (Q)  [0.000417834, 0.00782618, ..., 7.3443e-05, 7.86997]  [1.43196, 0.00113644, ..., 36833.5, 3808.16]
  energy_offset             float64            [meV]  (Q)  [-0.000249062, 0.000192105, ..., -0.000776911, 0.000175993]  [2.68261e-07, 2.35997e-07, ..., 3.36269e-07, 3.19884e-07]

We can also plot the parameters as a function of `Q` using the `plot_parameters` method.

In [20]:
# Plot some of fitted parameters as a function of Q
vanadium_analysis.plot_parameters(names=['DeltaFunction area'])

NameError: name 'vanadium_analysis' is not defined

In [ ]:
vanadium_analysis.plot_parameters(names=['Res. Gauss width'])

In [ ]:
vanadium_analysis.plot_parameters(names=['energy_offset'])

We are now happy with our resolution function and can start looking at the data we want to fit. We first load and inspect the data in the same way as before. 

In [ ]:
diffusion_experiment = Experiment('Diffusion')

file_path = pooch.retrieve(
    url='https://github.com/easyscience/dynamics-lib/raw/refs/heads/master/docs/docs/tutorials/data/diffusion_data_example.h5',
    known_hash='5fe846b19aacbda8b8b936eb2e5310d025dc56c25b0b353521e7d6b921f229ab',
)

diffusion_experiment.load_hdf5(filename=file_path)

In [ ]:
diffusion_experiment.plot_data(slicer=True)

The data seems to have a sharp elastic peak, a quasielastic peak and a non-zero background. We set up the corresponding `SampleModel` and `BackgroundModel` just like before.

In [ ]:
delta_function = DeltaFunction(display_name='DeltaFunction', area=0.2)
lorentzian = Lorentzian(display_name='Lorentzian', area=0.5, width=0.3)
component_collection = ComponentCollection(
    components=[delta_function, lorentzian],
)

sample_model = SampleModel(
    components=component_collection,
)

background_model = BackgroundModel(components=Polynomial(coefficients=[0.001]))

We also create a new instrument_model and attach it to our analysis. We want to use the resolutin that we determined from the vanadium data. Ideally, we'd just attach it to the instrument model like before. However, this is currently not possible due to a small issue in EasyDynamics that will be fixed asap. For now, we need to hack it a bit.

In [ ]:
instrument_model = InstrumentModel(
    background_model=background_model,
)

diffusion_analysis = Analysis(
    display_name='Diffusion Analysis',
    experiment=diffusion_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

# We need to hack in the resolution model from the vanadium analysis,
# since the setters and getters overwrite the model. This will be fixed
# asap.
diffusion_analysis.instrument_model._resolution_model = (
    vanadium_analysis.instrument_model.resolution_model
)

We don't want to fit our resolution anymore, so we fix all the parameters in it.

In [ ]:
# We fix all parameters of the resolution model.
diffusion_analysis.instrument_model.resolution_model.fix_all_parameters()

Before we start fitting it is a good idea to check how good our start guesses are. We do this by plotting the data and the model using `plot_data_and_model`:

In [ ]:
diffusion_analysis.plot_data_and_model()

The start guesses are not perfect, but they look good enough to start fitting. Let's give it a try!

In [ ]:
diffusion_analysis.fit(fit_method='independent')
diffusion_analysis.plot_data_and_model()

The fit looks good, so now we want to look at the most interesting fit parameters: the width and area of the Lorentzian. In later versions of EasyDynamics it will be possible to fit them to e.g. a DiffusionModel.

In [ ]:
# Let us look at the most interesting fit parameters
diffusion_analysis.plot_parameters(names=['Lorentzian width', 'Lorentzian area'])

There is a clear trend: the area is more or less constant, while the width seems to increase with `Q^2`. We therefore try and fit a Brownian translational diffusion model to the data. In this model, the scattering is given by
$$
I(Q,E) = S \frac{\Gamma(Q)}{\Gamma(Q)^2 + E^2},
$$
where $\Gamma(Q) = D Q^2$ and $D$ is the diffusion coefficient. $S$ is an overall scale.

In addition to this diffusion model, there is still the elastic incoherent scattering.

We create a new `SampleModel` which as a `DeltaFunction` component for the elastic incoherent scattering and a `BrownianTranslationalDiffusion` diffusion model to describe the rest. We also create a new `BackgroundModel` and `InstrumentModel`.

In [ ]:
delta_function = DeltaFunction(display_name='DeltaFunction', area=0.2)
component_collection = ComponentCollection(
    components=[delta_function],
)
diffusion_model = BrownianTranslationalDiffusion(
    display_name='Brownian Translational Diffusion', diffusion_coefficient=2.4e-9, scale=0.5
)

sample_model = SampleModel(
    components=component_collection,
    diffusion_models=diffusion_model,
)

background_model = BackgroundModel(components=Polynomial(coefficients=[0.001]))

instrument_model = InstrumentModel(
    background_model=background_model,
)

We attach all our models to a new `Analysis` object, again being mindful that we need to hack in the resolution. I promise this will be fixed soon!

In [ ]:
diffusion_model_analysis = Analysis(
    display_name='Diffusion Full Analysis',
    experiment=diffusion_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

# We again need to hack in the resolution model from the vanadium
# analysis, since the setters and getters overwrite the model. This will
# be fixed asap.
diffusion_model_analysis.instrument_model._resolution_model = (
    vanadium_analysis.instrument_model.resolution_model
)
diffusion_model_analysis.instrument_model.resolution_model.fix_all_parameters()

As always, we first check the start parameters before we fit

In [ ]:
diffusion_model_analysis.plot_data_and_model()

We can now fit all the data simultaneously to our model. It looks good!

In [ ]:
diffusion_model_analysis.fit(fit_method='simultaneous')
diffusion_model_analysis.plot_data_and_model()

It does not make sense to plot the diffusion parameters, but we can display them (with uncertainties) like this.

In [ ]:
diffusion_model.get_all_parameters()